## IMDB top 250 movies - data retrieval

This notebook should implement the web scraping of the [IMDB top 250 movies chart]( https://www.imdb.com/chart/top/).

Opening the link you should see the default chart view:



<p>
  <img src="assets/imdb_top_charts.png" style="width:800px;" alt="IMDB default view"/>
</p>

<br />

**Switch to the detailed view by clicking the detailed view button as shown in the picture above. Your view should now have all relevant information you need for scraping**

<br />

<p>
  <img src="assets/imdb_top_charts_detailed.png" style="width:800px;" alt="IMDB detailed view"/>
</p>


The expected result is a JSON file containing an array of 250 entries, i.e.:

```json
[
    {
        "title": "The Shawshank Redemption",
        "year": 1994,
        "rating": 9.3,
        "directors": [
            "Frank Darabont"
        ],
        "movie_stars": [
            "Tim Robbins",
            "Morgan Freeman",
            "Bob Gunton"
        ]
    },
    {
        "title": "The Godfather",
        "year": 1972,
        "rating": 9.2,
        "directors": [
            "Francis Ford Coppola"
        ],
        "movie_stars": [
            "Marlon Brando",
            "Al Pacino",
            "James Caan"
        ]
    },
    ...
]
```

### Notes
- Notice that the ranking numbers are removed in the JSON file (i.e. `1. The Shawshank Redemption` is saved as `"title": "The Shawshank Redemption"`)
- "year" is an integer
- "rating" is a floating number
- each movie can have multiple directors and/or movie stars - save them as an array of strings.
- as always, you are free to implement a different way of scraping the mentioned data - i.e. you can click on each movie in the chart and get the data from the movie's dedicated page.

### Relevant documentation links
- [Selenium for Python docs](https://selenium-python.readthedocs.io/)
- Standard Python libraries docs: [json](https://docs.python.org/3/library/json.html), [time](https://docs.python.org/3/library/time.html)


In [1]:
# Import libraries and define global variables/parameters
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
import json

URL = "https://www.imdb.com/chart/top/"
OUTPUT_JSON_FILE_NAME = "output.json"

### Navigating to the URL

Use the cell below to navigate to the url stored in the `URL` variable with Selenium

In [2]:
# navigate to the URL
driver = webdriver.Chrome()
driver.get(URL)
sleep(3)

### Switching to the detailed view of the IMDB top 250 chart
Use Selenium to find and click the relevant button to switch to detailed view of the chart.

In [3]:
# switch to detailed view which shows the movie stars too for each movie
driver.find_element(By.CSS_SELECTOR, "#list-view-option-detailed").click()
sleep(3)

### Selecting chart entries
Use your preferred method of selection to produce a list of 250 HTML elements, each corresponding to 1 chart entry (movie) with all the data you need to scrape about it.

In [4]:
movies_list = ...
movies_list = driver.find_elements(By.CSS_SELECTOR, "li.ipc-metadata-list-summary-item")
assert len(movies_list) == 250

Iterate over the produced list of HTML elements above, and scrape all needed data for each movie.

In [5]:
data = []
# do the magic (implement and store the data)
for movie_entry in movies_list:
    title = movie_entry.find_element(By.CSS_SELECTOR, "h3").text
    # remove the leaderboard number from the title
    first_space_in_title = title.index(" ")
    title = title[first_space_in_title+1:]

    year = movie_entry.find_elements(By.CSS_SELECTOR, "span.dli-title-metadata-item")[0].text
    rating = movie_entry.find_element(By.CSS_SELECTOR, "span.ipc-rating-star--rating").text

    directors = movie_entry.find_elements(By.CSS_SELECTOR, "a.dli-director-item")
    directors = [director.text for director in directors]
    movie_stars = movie_entry.find_elements(By.CSS_SELECTOR, "a.dli-cast-item")
    movie_stars = [moive_star.text for moive_star in movie_stars]

    data.append({
        "title": title,
        "year": int(year),
        "rating": float(rating),
        "directors": directors,
        "movie_stars": movie_stars
    })
driver.quit()

Save the scraped data to the JSON file (name and path defined in the `OUTPUT_JSON_FILE_NAME` variable at the top of this notebook).

In [6]:
with open(OUTPUT_JSON_FILE_NAME, "w") as f:
    json.dump(data, f, indent=4)

# produce a small subset of the data to "sample_output.json"
with open("sample_output.json", "w") as f:
    json.dump(data[:5], f, indent=4)